# Spatial accessibility and interaction through networks

In spatial data science, we use [spatial
graphs](../spatial_graphs/hands_on.qmd) to reflect the degree of spatial
interaction between observations. You already know how to represent
interaction based on contiguity, distance or spatial regimes. However,
in many contexts (notably urban ones), interaction follows spatially
restricted movement patterns, such as those along street networks. In
this session, we will cover movement through the streets as well as
movement by public transport.

In [ ]:
import datetime

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandarm
import r5py
import shapely

from libpysal import graph, kernels

We start with the data on census areas in Prague, as we did in the last
session. The dataset contains relatively granular spatial units with
information about the population living in each, based on 2021 data.

In [ ]:
prague = gpd.read_file(
    "https://martinfleischmann.net/sds/spatial_graphs/data/zsj_prague_2021.gpkg"
)
prague = prague.set_index("NAZ_ZSJ")
prague.explore()

## Walking through the streets

First, we will rely on the `pandarm` package to model interactions
reflecting movement (walking, driving and biking) through the streets.

All parts of the analysis here require you to build a routable network.
While you can do this manually from sets of nodes and edges (with edge
weights), the most common approach is to rely on the street network
defined by OpenStreetMap. Given a `GeoDataFrame`, `pandarm` can fetch
the data and construct such a network itself using
`pandarm.Network.from_gdf`, as shown below. However, this uses the
Overpass API, which is currently struggling to keep up with demand and
block bots. To avoid additional requests, the code below was used to
download, construct and save the network to an HDF5 file suitable for
multi-component data like this.

``` py
network = pandarm.Network.from_gdf(prague, network_type="walk")
network.save_hdf5("data/prague_walk.h5")
```

We can load the saved file directly with `pandarm`. Download the
[prague_walk.h5](https://martinfleischmann.net/sds/spatial_networks/data/prague_walk.h5)
file and put it in the `data` folder (or change the path below).

In [ ]:
network = pandarm.Network.from_hdf5("data/prague_walk.h5")

The network is, in principle, composed of two GeoDataFrames: one
capturing edges (streets) and another capturing nodes (intersections).

Edges:

In [ ]:
network.edges_df.head()

Nodes:

In [ ]:
network.nodes_df.head()

Nodes have a special role in `pandarm.Network`, as the analysis outcome
is typically linked to node geometry, as we’ll see below.

The second dataset used in this session captures locations of public and
private schools across Prague and comes from
[GeoPortalPraha.cz](https://geoportalpraha.cz/data-a-sluzby/5d32db18a5d545a79ad3029917984a89).
Read the data and familiarise yourself with it.

In [ ]:
schools = gpd.read_file(
    'https://martinfleischmann.net/sds/spatial_networks/data/schools.zip'
)
schools.head()

To work with the school locations within the network, we need to assign
them to the network using the `set_pois` (set Points of Interest)
method.

In [ ]:
network.set_pois(
    category="schools",
    maxdist=2000,
    maxitems=5,
    x_col=schools.geometry.x,
    y_col=schools.geometry.y,
)

### Nearest POIs

Using this, you can find the distances to the nearest schools. At most,
you can find the distances to the five nearest schools specified above.

In [ ]:
nearest_schools = network.nearest_pois(
    distance=2000,
    category="schools", 
    num_pois=5,
)
nearest_schools.head()

The distance is measured from every node in the network and can be
linked back to the `nodes_df`. That allows you to easily visualise the
outcome: a map of the distance to the nearest school (of any kind) when
walking along the street network.

In [ ]:
ax = network.nodes_df.plot(nearest_schools[1], markersize=.01, legend=True)
ax.set_axis_off()

For a better illustration, let’s use only a subset of specific schools.
Filter out `'Základní umělecká škola'`, which are art schools.

In [ ]:
art_school = schools.query("TYP_TXT == 'Základní umělecká škola'")
network.set_pois(
    category="art_school", 
    maxdist=5000,
    maxitems=1,
    x_col=art_school.geometry.x, 
    y_col=art_school.geometry.y,
)

Get the distances to the nearest art school. This time, also fetch the
ID of the PoI, not just the distance.

In [ ]:
nearest_art_school = network.nearest_pois(
    distance=5000, 
    category="art_school", 
    include_poi_ids=True,
)
nearest_art_school.head()

The resulting DataFrame now contains two columns: the first captures the
distance, and the second captures the ID of the nearest school. This ID
is an integer position, not the index.

In [ ]:
ax = art_school.plot(markersize=1, color='orange', zorder=2)
network.nodes_df.plot(
    nearest_art_school[1], markersize=.01, legend=True, ax=ax,
)
ax.set_axis_off()

The ID column can be used to allocate nodes to the nearest school, as
shown in the map below.

In [ ]:
ax = art_school.plot(markersize=1, color='black', zorder=2)
network.nodes_df.plot(
    nearest_art_school["poi1"], markersize=.01, categorical=True, cmap="Spectral", ax=ax,
)
ax.set_axis_off()

This is, in principle, a Voronoi tessellation based on distance along
the street network rather than typical Euclidean distance.

### Isochrones

Since we have started allocating space to locations based on distance,
let’s stick with this for a bit and explore isochrones. An isochrone
delineates an area reachable via the network within a set time or
distance. `pandarm` can help us define isochrones relatively easily
(though we need to do some manual coding).

We’re going to generate isochrones reflecting an approximate 15-minute
walking distance (~1200 metres) from each art school. To do so, we first
need to identify the node in the network nearest to each art school, as
the analysis is tied to existing nodes.

In [ ]:
art_schoool_nodes = network.get_node_ids(
    art_school.geometry.x, 
    art_school.geometry.y,
)
art_schoool_nodes.head()

The resulting series gives us the ID of the node used to represent the
approximate location of each art school. With this, you can find all
other nodes in the specified range, which represents an isochrone.

In [ ]:
isochrones = network.nodes_in_range(art_schoool_nodes, radius=1200)
isochrones

The output DataFrame is not the most user-friendly, but you can see that
the `source` column contains repeated IDs of source nodes, along with
all other nodes within the `radius` and the exact distance to them.
Getting the isochrones as polygons is then a simple GeoPandas operation.

In [ ]:
isochrones_poly = (
    network.nodes_df.loc[isochrones["destination"]]
    .dissolve(isochrones["source"].values)
    .concave_hull(ratio=0.2)
)
isochrones_poly.head()

You can now explore the outcome.

In [ ]:
m = isochrones_poly.explore()
art_school.explore(m=m, color='red')

If that is of interest, you can further analyse and dissect the nodes
forming the isochrones, as each has a distance. For example, you can
peel off outer nodes to form smaller isochrones without recomputing the
routes.

In [ ]:
ax = network.nodes_df.loc[isochrones["destination"]].plot(isochrones['length'].values, markersize=.1, legend=True)
ax.set_axis_off()

### Network-based aggregations

`pandarm` is also well suited for data aggregation along the network.
This can capture accessible population, accessible school capacity or
similar measures. Our school dataset contains information about school
capacity in the `KAPACITA` column, so we can try to understand how
capacity affects access.

We also have population information linked to census output areas, so we
can ask how school capacity compares with the local population.

First, we need to assign variables to the network. Since we cannot add
additional nodes to the network, we find the nearest existing node to
each point with a value and assign the values to those nodes.

In [ ]:
art_node_ids = network.get_node_ids(art_school.geometry.x, art_school.geometry.y)
pop_node_ids = network.get_node_ids(prague.centroid.x, prague.centroid.y)

network.set(art_node_ids, variable=art_school["KAPACITA"], name="art_school_capacity")
network.set(pop_node_ids, variable=prague["n_people"], name="population")

We are interested in the number of places in art schools within an
accessible range per 1,000 people. By accessible, we mean approximately
30 minutes of walking, as people are more willing to walk further for
specialised needs like this one. However, the further away a school is,
the less willing people are to make the trip. We can therefore start by
allocating accessible population to each node as a distance-weighted
sum.

In [ ]:
accessible_population = network.aggregate(
    distance=2400,
    func="sum",
    decay="linear",
    name="population",
)
accessible_population.head()

This aggregation results in a spatial distribution of accessible
population that highlights areas of high population density, but is
relatively smooth.

In [ ]:
ax = network.nodes_df.plot(accessible_population, markersize=.01, legend=True)
ax.set_axis_off()

We can now do the same aggregation for the capacity of art schools,
again using linear decay to account for the difference in willingness to
walk as distance increases.

In [ ]:
accessible_capacity = network.aggregate(
    distance=2400, 
    func="sum", 
    decay="linear", 
    name="art_school_capacity",
)

The accessible capacity then takes into account the locations of art
schools, their individual capacities and the ability to reach them via
the street network.

In [ ]:
ax = art_school.plot(color='red', zorder=2, markersize=2)
ax = network.nodes_df.plot(accessible_capacity, markersize=.01, legend=True, ax=ax)
ax.set_axis_off()

Now, we can simply normalise the accessible capacity by the accessible
population to get an approximation of general art school accessibility.

In [ ]:
normalised_capacity = (
    accessible_capacity / accessible_population
).replace(np.inf, np.nan) * 1000

The results show hotspots of higher accessibility and coldspots of lower
accessibility, with some peripheral locations indicating high
accessibility. Those schools are quite likely serving a wider suburban
region.

In [ ]:
ax = art_school.plot(color='red', zorder=2, markersize=2)
network.nodes_df.plot(normalised_capacity, markersize=.01, legend=True, vmax=100, ax=ax)
ax.set_axis_off()

Keep in mind that this example is an oversimplification, as people will
often travel across the city using public transport, which significantly
changes the perception of distance, as we will explore below.

### Network interaction graphs

The idea of accessibility from one place to another through the street
network can be translated into spatial weights encoding spatial
interaction. `libpysal` has a built-in method to derive such weights
from a provided `pandarm` network.

In [ ]:
# this needs new libpysal
travel_cost_graph = graph.Graph.build_travel_cost(
    prague.centroid,
    network=network,
    threshold=2400,
    kernel="bisquare",
)
travel_cost_graph

The resulting `Graph` can be used like any other, for example, to derive
a spatial lag.

In [ ]:
lagged_population = travel_cost_graph.lag(prague["n_people"])
prague.explore(lagged_population)

> **Additional information**
>
> `pandarm` is a *fork* of another package named `pandana`, which was
> abandoned for years (but is now in maintenance mode). `pandarm`
> contains some additional functionality, but resources that refer to
> `pandana` also apply to `pandarm`. Simply replace `pandana` in the
> code with `pandarm`.
>
> For more details, you can check the documentation for
> [pandarm](https://oturns.github.io/pandarm/) and
> [pandana](https://udst.github.io/pandana/), and the [chapter on
> Accessibility](https://knaaptime.com/urban_analysis/04_access/simple_access.html)
> from Eli Knaap’s book *Urban Analysis & Spatial Science*.

## Multi-modal movement

Up until now, we have assumed movement through the street network on
foot. However, urban environments are often served by dense public
transport networks, some of which run on streets and some of which do
not. To provide a more realistic estimate of accessibility and
network-based interaction, we shall combine different modes of transport
and make full use of public transport. To do so, we need data capturing
public transport schedules in a format called the [General Transit Feed
Specification (GTFS)](https://gtfs.org). This contains information about
routes, stops, schedules, transfers, agencies running individual routes
and other aspects of public transport systems. For Prague, you [can
get](https://pid.cz/o-systemu/opendata/) this information for two weeks
ahead.

`pandarm` cannot model multimodal movement, but we can use another tool:
`r5py`. It works similarly to `pandarm` by creating a network object
first. In addition to the GTFS data, you need to provide a local extract
from OpenStreetMap for `r5py` to use as its street network
representation for routing.

In [ ]:
transport_network = r5py.TransportNetwork(
    osm_pbf="https://martinfleischmann.net/sds/spatial_networks/data/praha-261020.osm.pbf",
    gtfs=["https://martinfleischmann.net/sds/spatial_networks/data/PID_GTFS.zip"],
    allow_errors=True,
)

### Isochrones

As with `pandarm`, we can delineate isochrones as areas accessible
within a given time from one or more origins. The two packages work a
bit differently, so their APIs do not match. Nevertheless, the concepts
are similar. In `r5py`, generating isochrones is easier, as there is a
dedicated method for doing so: `r5py.Isochrones`.

Let’s try to define isochrones around art schools in Prague as we did
before, this time using public transport information.

In [ ]:
art_school['id'] = art_school.NAZEV_ZAR

isochrones = r5py.Isochrones(
    transport_network,
    origins=art_school,
    departure=datetime.datetime(2026, 10, 22, 10, 30),
    isochrones=[5, 10, 15],
)
isochrones

The output is a MultiLineString for each time, providing the boundaries
of each isochrone.

In [ ]:
isochrones['travel_time'] = isochrones['travel_time'].astype(int)
ax = isochrones.plot(
    'travel_time', legend=True, legend_kwds={"label": "Travel time [seconds]"},
)
ax.set_axis_off()

As you can see above, the isochrones are joined together: we get areas
accessible from one or more art schools, not a boundary for each school.
It is often easier to work with isochrones as polygons, so let’s build
areas from these LineStrings.

In [ ]:
isochrones["polygons"] = shapely.build_area(
    isochrones.geometry 
).set_crs(isochrones.crs)
isochrones.set_geometry("polygons").explore('travel_time')

### Travel-time matrix

The main use case for `r5py` is computing travel-time matrices. A
travel-time matrix contains, as the name suggests, travel times through
the network between all pairs of provided origins and destinations. We
can illustrate this by computing travel times from all the census areas
in our `prague` dataset to Albertov.

In [ ]:
prague['id'] = prague.index

origins = prague.set_geometry(prague.centroid)
destinations = origins.loc[["Albertov"]]

travel_times = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins,
    destinations=destinations,
    departure=datetime.datetime(2026, 10, 22, 10, 30),
    snap_to_network=True,
)
travel_times

Here, the result shows the time needed to reach Albertov from across the
city via public transport.

In [ ]:
ax = prague.plot(
    travel_times.travel_time.values,
    legend=True,
)
ax.set_axis_off()

However, this can be built as a full $N \times N$ matrix, computing
travel times between all pairs of census areas.

In [ ]:
travel_times = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins,
    destinations=origins,
    departure=datetime.datetime(2026, 10, 22, 10, 30),
    snap_to_network=True,
    max_time=datetime.timedelta(minutes=30),
)
travel_times

Given this is a full $N*N$ matrix, we can directly convert it to
`Graph`.

In [ ]:
transit_graph = graph.Graph.from_adjacency(
    adjacency=travel_times.dropna(),
    focal_col="from_id",
    neighbor_col="to_id",
    weight_col="travel_time",
)
transit_graph

With this, we already know how to work with the graph.

In [ ]:
transit_graph["Albertov"]

One issue that remains is that the weight is represented in minutes. We
can use the `kernels` module of `libpysal` to convert it into a useful
weight before generating the `Graph`.

In [ ]:
travel_times["weights"] = kernels.kernel(
    travel_times["travel_time"], 
    bandwidth=30, 
    kernel="bisquare",
    decay=True,
)
travel_times

The code above converts a travel time of 0 to a weight of 1 and a travel
time of 30 to a weight of 0, with a bisquare kernel defining all values
in between. You can then use the new column as the weight column when
constructing the `Graph`.

In [ ]:
transit_graph_weighted = graph.Graph.from_adjacency(
    adjacency=travel_times.dropna(),
    focal_col="from_id",
    neighbor_col="to_id",
    weight_col="weights",
)

You can see that the weights are now as expected for spatial modelling
tasks.

In [ ]:
transit_graph_weighted["Albertov"]

This can be nicely illustrated on a map. Notice the discontinuity caused
by the locations of public transit stops.

In [ ]:
ax = prague.assign(weight=transit_graph_weighted["Albertov"]).plot(
    'weight', legend=True, tiles=True, alpha=.7,
)
ax.set_axis_off()

See the effects of the spatial graph defined in this way on the spatial
lag of population.

In [ ]:
lagged_population = transit_graph_weighted.lag(prague["n_people"])

prague.explore(lagged_population)

The main public transport routes are noticeable in the result.

> **Additional reading**
>
> For more resources on using `r5py`, you can check the
> [documentation](https://r5py.readthedocs.io/stable/index.html) or the
> [tutorial](https://sustainability-gis.readthedocs.io/en/latest/tutorials/r5py_demo.html)
> written by Henrikki Tenkanen, Christoph Fink and Willem Klumpenhouwer.